# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do NOT treat as dict

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their fields by `@id`
record_sets = list(dataset.record_sets)

record_set_ids = []
for rec in record_sets:
    print(f"RecordSet @id: {rec.id}")
    record_set_ids.append(rec.id)
    print(f"  Name: {rec.name}")
    print(f"  Description: {rec.description}")
    print(f"  Fields:")
    for field in rec.fields:
        print(f"    Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print('-' * 60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this FAIR2 dataset, there is typically one record set containing the main table.
# We'll extract all record sets and load as DataFrames. You can select by @id as shown above.
dfs = {}
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    dfs[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(df.head(2))
    print("-")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose the main record set for EDA (using the first RecordSet for illustrative purposes)
main_record_set_id = record_set_ids[0]
df = dfs[main_record_set_id]

# Show all available columns
print("Available columns:", df.columns.tolist())

# Let's try to filter on a numeric column e.g., "MW/kDa" or "unique_peptides" or "abundance_norm_B" if present
# Use actual field/column names from the dataset overview output
# We'll try guessing popular mass spec fields
numeric_candidates = [col for col in df.columns if any(key in col.lower() for key in ['mw', 'peptide', 'abundance', 'mass', 'coverage', 'count'])]
print("Numeric candidates:", numeric_candidates)

# Pick the first possible numeric field for demo
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = df.columns[0]  # fallback
print(f"Using numeric field for analysis: {numeric_field}")

if pd.api.types.is_numeric_dtype(df[numeric_field]):
    threshold = df[numeric_field].mean()  # use mean as threshold for demo
    filtered_df = df[df[numeric_field] > threshold]
else:
    # Try to convert if stored as string
    filtered_df = df.copy()
    filtered_df[numeric_field + '_num'] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
    threshold = filtered_df[numeric_field + '_num'].mean()
    filtered_df = filtered_df[filtered_df[numeric_field + '_num'] > threshold]
    numeric_field = numeric_field + '_num'

print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a category field, e.g., "accession" or "category" if present
cat_candidates = [col for col in df.columns if any(key in col.lower() for key in ['accession', 'type', 'sample', 'description', 'id', 'category', 'species']) and col != numeric_field]
print("Categorical candidates:", cat_candidates)

if cat_candidates:
    group_field = cat_candidates[0]
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram of numeric field
plt.figure(figsize=(7, 4))
filtered_df[numeric_field].hist(bins=30, color='skyblue', edgecolor='black')
plt.title(f"Distribution of {numeric_field} (filtered)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouped_df exists, plot group means
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 5))
    plt.bar(grouped_df[group_field].astype(str), grouped_df[numeric_field])
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.